# Validate DC powerflow vs PTDF (battery_bands)

Dieses Notebook validiert die elektrotechnische Logik in `src/battery_bands.py`.

**Validierungsidee:**
- Wir lassen `battery_bands` dieselben internen Objekte bauen (Supernetz, B-Matrix, PTDF je Komponente).
- Wir vergleichen für eine kleine Einspeiseänderung ΔP am BESS:
  - ΔF aus **DC-Loadflow** (solve θ → Flüsse)
  - ΔF aus **PTDF** (linear sensitivities)

Wenn beide übereinstimmen (bis auf numerisches Rauschen), ist die PTDF-Berechnung konsistent zur DC-Solve-Logik – also genau die Rechenlogik, die im Service später verwendet wird.


### Imports

In [ ]:
import inspect
import numpy as np
import pandas as pd
from pathlib import Path
import src.battery_bands as bb

### Run bb.run() in debug mode (no IO)

In [22]:
GRAPH_PATH = "src/data/raw/graph/whole_graph.json"  

debug = bb.run(
    return_debug=True,
    debug_max_timestamps=1,
    graph_path_override=GRAPH_PATH,
    pred_dir_override=None,    # keine Forecasts im Validation-Graph
    out_dir_override=None,     # nichts schreiben
    disable_io=True,           # kein CSV schreiben
)

print("debug keys:", sorted(debug.keys()))

Config: UTIL_TARGET_PCT=50.0% (scale=0.500)


debug keys: ['PTDF_by_comp', 'bands_df', 'basecase_violations', 'battery_infos', 'cfg', 'components', 'disable_io', 'edge_meta', 'flows_now', 'graph_path', 'lines', 'out_dir', 'pred_dir', 'slack_by_comp', 'super_nodes', 'timestamps', 'util_df']


### Ziel: Reproduzierbarer Validierungs-Run ohne I/O

In diesem Schritt wird `battery_bands.run()` im **Debug-Modus** auf ausgeführt.  
Wichtig: Es werden **keine Forecast-Dateien geladen** und **keine CSVs geschrieben** (`disable_io=True`). Stattdessen liefert die Funktion ein `debug`-Dictionary zurück, das alle internen Zwischenergebnisse enthält, die wir für die Validierung brauchen.

#### Was enthält `debug`?
Die ausgegebenen Keys zeigen, dass wir Zugriff auf die zentralen Bausteine der Berechnung haben:

- `super_nodes`, `lines`, `edge_meta`: Supernetz-Struktur und Leitungsparameter
- `components`, `slack_by_comp`: Netztopologie in Komponenten + gewählter Slack
- `PTDF_by_comp`: PTDF-Matrix (sensitivitätsbasiertes Modell)
- `flows_now`, `basecase_violations`: DC-Load-Flow Ergebnis und evtl. Basecase-Grenzverletzungen
- `bands_df`, `util_df`: daraus abgeleitete Leistungsbänder und Leitungs-Auslastungen (nur im Speicher)


---

### Helper: robust call to solve_dc_flows_per_timestamp

In [23]:
def solve_dc(bb_module, **kwargs):
    fn = bb_module.solve_dc_flows_per_timestamp
    sig = inspect.signature(fn)
    allowed = set(sig.parameters.keys())
    call_kwargs = {k: v for k, v in kwargs.items() if k in allowed}
    return fn(**call_kwargs)

Diese Hilfsfunktion ruft den DC-Loadflow direkt über die produktive Funktion aus `battery_bands.py` auf.  
Über `inspect.signature` werden nur die tatsächlich unterstützten Parameter übergeben, sodass das Notebook robust bleibt.  
Damit validieren wir den DC-Loadflow ohne eigene Rechenlogik zu duplizieren.


### Build internals

In [28]:
cfg_dump = debug.get("cfg", {}) or {}
X_EPS_OHM      = float(cfg_dump.get("X_EPS_OHM", bb.cfg.X_EPS_OHM))
S_BASE_MVA     = float(cfg_dump.get("S_BASE_MVA", bb.cfg.S_BASE_MVA))
V_KV_DEFAULT   = float(cfg_dump.get("V_KV_DEFAULT", bb.cfg.V_KV_DEFAULT))
COSPHI_MIN     = float(cfg_dump.get("COSPHI_MIN", bb.cfg.COSPHI_MIN))
SLACK_NODE_ID  = str(cfg_dump.get("SLACK_NODE_ID", bb.cfg.SLACK_NODE_ID))
UTIL_TARGET_PCT = float(cfg_dump.get("UTIL_TARGET_PCT", bb.cfg.UTIL_TARGET_PCT))
util_scale = UTIL_TARGET_PCT / 100.0

super_nodes = debug["super_nodes"]

have_raw = ("components_raw" in debug) and ("comp_cache_raw" in debug) and ("PTDF_by_comp_raw" in debug)

if have_raw:
    components = debug["components_raw"]
    comp_cache = debug["comp_cache_raw"]
    PTDF_by_comp_raw = debug["PTDF_by_comp_raw"]
    edge_meta = debug.get("edge_meta", {})
    lines = debug.get("lines", [])
else:
    nodes, edges, node_types, node_features = bb.load_graph(Path(GRAPH_PATH))
    rep_map, super_nodes2, members_by_rep, super_edges, n_contracted, n_kept = bb.contract_graph(nodes, edges, X_EPS_OHM)

    # sanity: must match debug
    if list(super_nodes2) != list(super_nodes):
        print("WARN: super_nodes from debug != rebuilt; using rebuilt version.")
        super_nodes = super_nodes2

    B, node_index, lines, edge_meta = bb.build_B_and_lines_pu(
        super_nodes=super_nodes,
        super_edges=super_edges,
        V_kV_default=V_KV_DEFAULT,
        S_base_MVA=S_BASE_MVA,
    )

    components, slack_by_comp, comp_data = bb.build_components_and_comp_data(
        super_nodes=super_nodes,
        lines=lines,
        B=B,
        node_index=node_index,
        slack_fixed=SLACK_NODE_ID,
    )

    PTDF_by_comp_raw = bb.compute_ptdf_by_component(
        components=components,
        comp_data=comp_data,
        lines=lines,
    )

    comp_cache = bb.build_component_cache(
        components=components,
        comp_data=comp_data,
        PTDF_by_comp=PTDF_by_comp_raw,
        edge_meta=edge_meta,
        node_index=node_index,
        V_kV_default=V_KV_DEFAULT,
        cosphi_min=COSPHI_MIN,
        util_scale=util_scale,
    )

print("n super_nodes:", len(super_nodes))
print("n components:", len(components))


n super_nodes: 11
n components: 1


Hier werden die relevanten Konfigurationsparameter aus dem Debug-Output übernommen, um identische Randbedingungen wie im produktiven Lauf sicherzustellen.  
Anschließend werden Komponenten, PTDFs und Caches ausschließlich mit Funktionen aus `battery_bands.py` rekonstruiert.  
So ist gewährleistet, dass die Validierung exakt auf derselben Logik basiert wie die eigentliche Leistungsband-Berechnung.


### PTDF-basierte Sensitivität einer Einheitsinjektion

In [25]:
comp = components[0]
ck = frozenset(comp)

ptd = PTDF_by_comp_raw[ck]
PTDFm = ptd["PTDF"]          # (m, k)
line_ids = ptd["line_ids"]   # len m
pos = ptd["pos"]             # node -> col (non-slack only)

print("PTDF shape:", PTDFm.shape, "m lines:", len(line_ids), "k non-slack:", len(pos))

if len(pos) == 0:
    raise RuntimeError("Komponente hat k=0 (nur Slack). Nimm eine andere Komponente: components[i].")

test_node = next(iter(pos.keys()))
print("test_node:", test_node)

P_non_slack = np.zeros(PTDFm.shape[1], dtype=float)
P_non_slack[pos[test_node]] = 1.0

F_ptdf = PTDFm @ P_non_slack
ptdf_series = pd.Series(F_ptdf, index=line_ids, name="F_ptdf_MW")
ptdf_series.head(10)

PTDF shape: (11, 10) m lines: 11 k non-slack: 10
test_node: BOLN_E02


110-SHUW-WEDI-ROT,BOLN,SIES,SIEV SHUW-BOLN A3      -7.965238e-01
110-SHUW-WEDI-ROT,BOLN,SIES,SIEV BOLS A3-BOLN A3   -7.965238e-01
110-SHUW-WEDI-ROT,BOLN,SIES,SIEV BOLN A3-SIES A3   -2.034762e-01
110-SHUW-WEDI-ROT,BOLN,SIES,SIEV SIES A3-SIES      -1.479228e-17
110-SHUW-WEDI-ROT,BOLN,SIES,SIEV SIEV A3-SIES A3    2.034762e-01
Name: F_ptdf_MW, dtype: float64

Hier wird eine **+1 MW Einspeisung** am Knoten *BOLN_E02* angenommen (alle anderen 0).  
Die PTDF-Werte zeigen, **wie sich diese Einspeisung auf die einzelnen Leitungen verteilt**.

- Beträge wie **−0.80 MW** und **−0.20 MW** bedeuten: Die Leistung teilt sich auf mehrere Pfade auf.
- Werte nahe **0** (≈ 1e-17) sind numerisch vernachlässigbar.
- Das Vorzeichen beschreibt die **Flussrichtung** relativ zur Leitungsdefinition.


### DC-Lastfluss mit + 1 MW Einspeisung 

In [26]:
P_row = pd.Series(0.0, index=super_nodes, dtype=float)
P_row.loc[test_node] = 1.0

theta_global = np.zeros(len(super_nodes), dtype=float)

out = solve_dc(
    bb,
    components=components,
    comp_cache=comp_cache,
    edge_meta=edge_meta,
    P_row=P_row,
    S_base_MVA=S_BASE_MVA,
    V_kV_default=V_KV_DEFAULT,
    cosphi_min=COSPHI_MIN,
    util_scale=util_scale,
    theta_global=theta_global,
)

if isinstance(out, tuple) and len(out) == 2:
    flows_dc, basecase_violations = out
else:
    flows_dc, basecase_violations = out, []

dc_series = pd.Series(flows_dc, name="F_dc_MW").reindex(line_ids).fillna(0.0)

print("n DC flows:", len(flows_dc))
print("basecase_violations:", basecase_violations[:5], "..." if len(basecase_violations) > 5 else "")

cmp = pd.DataFrame({"F_dc_MW": dc_series, "F_ptdf_MW": ptdf_series})
cmp["abs_err"] = (cmp["F_dc_MW"] - cmp["F_ptdf_MW"]).abs()
cmp["rel_err"] = cmp["abs_err"] / (cmp["F_dc_MW"].abs() + 1e-12)

cmp.sort_values("abs_err", ascending=False).head()

n DC flows: 11
basecase_violations: [] 


,F_dc_MW,F_ptdf_MW,abs_err,rel_err
"110-SHUW-WEDI-ROT,BOLN,SIES,SIEV BOLN A3-SIES A3",-2.034762e-01,-2.034762e-01,1.609823e-15,7.911607e-15
"110-SHUW-WEDI-ROT,BOLN,SIES,SIEV SIES A3-SIES",-1.239803e-15,-1.479228e-17,1.225010e-15,1.223493e-03
"110-SHUW-WEDI-ROT,BOLN,SIES,SIEV BOLS A3-BOLN A3",-7.965238e-01,-7.965238e-01,1.110223e-16,1.393835e-16
"110-SHUW-WEDI-ROT,BOLN,SIES,SIEV SIEV A3-SIES A3",2.034762e-01,2.034762e-01,1.110223e-16,5.456281e-16
"110-SHUW-WEDI-ROT,BOLN,SIES,SIEV WEDI-SIEV A3",-2.034762e-01,-2.034762e-01,1.110223e-16,5.456281e-16


Hier berechnen wir dieselbe **Einheits-Einspeisung (+1 MW an `test_node`)** einmal mit dem **DC-Loadflow** (über `solve_dc_flows_per_timestamp`) und vergleichen die resultierenden Leitungsflüsse mit den **PTDF-Flüssen**.

Ergebnis: **DC und PTDF stimmen praktisch überein** (11 Flüsse, keine Basecase-Verletzungen).  
Die größten Abweichungen liegen im Bereich von **numerischem Rundungsrauschen** (≈ 1e−15). Der auffälligere `rel_err` entsteht dort, wo der echte Fluss **nahe 0** ist (Division durch sehr kleine Werte).


### Zusammenfassung der Validierung (DC-Powerflow vs. PTDF)

In [ ]:
max_abs = float(cmp["abs_err"].max())
max_rel = float(cmp["rel_err"].max())

print("max abs err:", max_abs)
print("max rel err:", max_rel)


max abs err: 1.609823385706477e-15
max rel err: 0.0012234933414879382


Die maximalen Abweichungen zwischen DC-Loadflow und PTDF sind extrem klein.  
Der **absolute Fehler (~1e−15)** liegt klar im Bereich numerischer Rundungsfehler.  
Der **relative Fehler (~1e−3)** tritt nur bei Leitungen mit **nahe-null Flüssen** auf und ist physikalisch unkritisch.

**Fazit:** Die PTDF-Implementierung ist konsistent mit dem DC-Powerflow und damit korrekt für die nachgelagerte Leistungsband-Berechnung.